In [ ]:
# ==============================================================
# Imports
# ==============================================================
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# ==============================================================
# Configuration
# ==============================================================
img_height, img_width = 224, 224
batch_size = 32
validation_split = 0.2
data_dir = '/kaggle/input/datasets/anandjain1112/3-crops-fruit-and-leaf-disease-dataset/3 Fruit and leaf crops disease dataset'
epochs = 50

# ==============================================================
# Load Dataset (Train / Validation Split)
# ==============================================================
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=validation_split,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=validation_split,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Class Names:", class_names)

# ==============================================================
# Normalization & Performance Optimization
# ==============================================================
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# ==============================================================
# Model: MobileNetV2 + 0.8 Dropout
# ==============================================================
base_model = tf.keras.applications.DenseNet121(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = True  # Freeze backbone

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),   # 🔥 80% Dropout
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# ==============================================================
# Compile Model
# ==============================================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ==============================================================
# Train Model
# ==============================================================
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=epochs
)

# ==============================================================
# Evaluate Model
# ==============================================================
train_loss, train_acc = model.evaluate(train_ds)
test_loss, test_acc = model.evaluate(test_ds)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

# ==============================================================
# Save Model
# ==============================================================
model.save('citrus_mobilenetv2_dropout08.h5')

# ==============================================================
# Plot Accuracy & Loss
# ==============================================================
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.show()

# ==============================================================
# Confusion Matrix & Classification Report
# ==============================================================
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))
